# Costruzione del panel

Questo notebook costruisce il panel di startup a partire dai CSV grezzi di
**PitchBook** e documenta ogni passaggio. È la traduzione in Python di due
script R scritti da terzi (`src/RCode/1_Arrange_DB.R`, 1257 righe, e
`src/RCode/2_Arrange_Final.R`, 269 righe), più i due passaggi di
post-elaborazione che li seguivano.

**L'obiettivo di questa fase è la riproduzione fedele, bug compresi.** Ogni
difetto noto dell'R viene riprodotto per default e sta dietro a un flag
`fix_*` in `PanelConfig` che vale `False`, dove `False` significa «fai quello
che faceva l'R». Correggerli è una fase successiva: prima si dimostra di aver
tradotto bene, poi si cambia.

Il criterio di successo non è dichiarativo. Gli script R salvavano dei file
intermedi, e quei file sono disponibili in `data/reference/`: ogni stadio viene
confrontato **colonna per colonna, riga per riga** con il file che deve
riprodurre. Sei confronti indipendenti, allineati per chiave e non per
posizione.


## Come si legge e si esegue

Ogni stadio ha tre parti:

1. una spiegazione di **cosa fa** e a quali righe dell'R corrisponde;
2. le **trappole**: i punti in cui una traduzione plausibile divergerebbe in
   silenzio, con il numero di righe che sbaglierebbe;
3. il **codice** che lo esegue, un'**ispezione** del risultato e, dove esiste
   un file di riferimento, il **rapporto di verifica**.

Gli stadi comunicano tramite file parquet in `data/interim/`, non tramite
memoria. Questo ha due conseguenze pratiche: si può **riprendere da qualsiasi
stadio** senza rifare i precedenti, e si può **riavviare il kernel** in
qualsiasi momento senza perdere niente. Se un checkpoint segnala un problema,
si corregge il codice, si riesegue solo quello stadio e si riverifica.

Niente in questo notebook scrive in `data/raw/` o in `data/reference/`: sono
gli input e la verità di riferimento, e restano intatti. Il panel finito
finisce in `data/interim/panel.csv.gz`.

I rapporti di verifica girano in un **sottoprocesso**. Non è un vezzo: leggere
`db_master_2.csv` come testo occupa qualche gigabyte, e tenerlo nel kernel per
il resto della sessione fa saltare la memoria a metà pipeline. Per lo stesso
motivo le celle di ispezione rileggono da parquet solo le colonne che servono e
liberano le variabili quando ha finito: il picco della pipeline è di circa
5,6 GB, e il margine non è grande.


In [ ]:
import gc
import subprocess
import sys
from pathlib import Path

import polars as pl

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.panel import (
    stage1_company,
    stage2_team,
    stage3_relations,
    stage4_deals,
    stage5_final,
    stage6_panel,
    stage7_competitors,
)
from src.panel.config import BUG_FLAGS, PanelConfig
from src.panel.validate import CHECKPOINTS

cfg = PanelConfig()

pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(40)

attive = cfg.active_fixes()
print("correzioni attive:", list(attive) if attive else "nessuna (comportamento R)")
print("flag disponibili :", len(BUG_FLAGS))
print("input grezzi     :", cfg.raw_dir)
print("riferimenti      :", cfg.ref_dir)
print("output di stadio :", cfg.interim_dir)


### Due funzioni di servizio

`verifica` lancia un checkpoint in un sottoprocesso e stampa il rapporto.
`carica` rilegge l'output di uno stadio da parquet, opzionalmente solo alcune
colonne, per ispezionarlo senza tenersi in memoria tutto il resto.


In [ ]:
def verifica(lettera: str) -> None:
    """Esegue un checkpoint in un sottoprocesso e ne stampa il rapporto."""
    cp = CHECKPOINTS[lettera]
    print(f"checkpoint {lettera}: {cp.interim}  contro  {cp.reference}")
    print(f"righe attese: {cp.expect_rows:,}   chiave: {cp.key}\n")
    esito = subprocess.run(
        [sys.executable, "-m", "src.panel.validate", "--checkpoint", lettera],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    print(esito.stdout or esito.stderr)


def carica(nome: str, colonne: list[str] | None = None) -> pl.DataFrame:
    """Rilegge l'output di uno stadio, proiettato alle colonne richieste."""
    return pl.read_parquet(cfg.interim(f"{nome}.parquet"), columns=colonne)


## Panoramica: sette stadi, sei checkpoint

| stadio | cosa aggiunge | tabelle grezze lette | checkpoint |
|---|---|---|---|
| 1 | sezione trasversale delle aziende e **scheletro del panel** | `Company`, `CompanyAffiliateRelation` | — |
| 2a | tabella **persona-azienda**: istruzione, esperienza, permanenza | `CompanyBoardTeamRelation`, `Person`, `PersonEducationRelation`, `PersonPositionRelation` | **A** |
| 2b | le colonne di **team** anno per anno | — | — |
| 3a | le colonne **competitor** (statiche) | `CompanySimilarRelation` | **B** |
| 3b | dipendenti, financials, news | `CompanyEmployeeHistoryRelation`, `CompanyFinancialRelation`, `CompanyNewsRelation` | — |
| 4 | **deal e investitori** | `Deal`, `DealInvestorRelation`, `Investor` | — |
| 5 | cumulate, `GrowthStage`, attributi del CEO, selezione colonne | — | **C**, **D** |
| 6 | raggruppamento degli stadi e **troncamento** all'uscita | — | **E** |
| 7 | competitor **temporizzati** anno per anno | `CompanySimilarRelation`, `Company` | **F** |

Gli stadi 1–5 traducono l'R. Gli stadi 6 e 7 traducono due passaggi che
venivano dopo: il primo non aveva codice sopravvissuto e le sue regole sono
state ricostruite dal file che produceva, il secondo veniva da un notebook
separato.

### Il database SQLite dell'R non serve

Gli script R caricavano 51 CSV in un database SQLite temporaneo e vi accedevano
per **indice posizionale** (`tbl[1]`, `tbl[17]`, ...). L'unica informazione che
quel database forniva era la corrispondenza fra indice e tabella, che è
l'ordine alfabetico dei file:

| idx | tabella | idx | tabella |
|----:|---|----:|---|
| 1 | `Company` | 19 | `Deal` |
| 2 | `CompanyAffiliateRelation` | 22 | `DealInvestorRelation` |
| 3 | `CompanyBoardTeamRelation` | 32 | `Investor` |
| 6 | `CompanyEmployeeHistoryRelation` | 43 | `Person` |
| 8 | `CompanyFinancialRelation` | 48 | `PersonEducationRelation` |
| 14 | `CompanyNewsRelation` | 49 | `PersonPositionRelation` |
| 17 | `CompanySimilarRelation` | | |

La versione Python legge i CSV direttamente, per nome.


## Passo 0 — l'estrazione è quella giusta?

I file in `data/reference/` sono la verità di riferimento. Se i CSV grezzi non
sono lo stesso *vintage* — la stessa data di scarico da PitchBook — nessun
checkpoint può passare, e il problema non sarebbe nel codice.

Il controllo costa pochi secondi e va fatto **prima** di tutto il resto:
confronta il numero di aziende con `YearFounded > 1999` (deve essere
esattamente 116.920) e verifica che nessuna azienda o coppia
azienda-persona presente nei riferimenti manchi dai grezzi.


In [ ]:
print(
    subprocess.run(
        [sys.executable, "scripts/check_extraction.py", "data/raw/pitchbook"],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    ).stdout
)


---
## Stadio 1 — aziende, affiliate e scheletro del panel

**Corrisponde a** `1_Arrange_DB.R:37-238`.

Legge `Company.csv` (134.355 aziende), ne tiene 39 colonne, e:

1. **sostituisce i placeholder di missing con NA veri.** L'R lo fa con un ciclo
   su tutte le colonne che cerca `""`, `"NA"`, `"N/A"`, `"NULL"`, `"NaN"`. Il
   momento in cui questo ciclo gira conta, come si vede subito sotto.
2. **crea quattro flag di presenza** — `Website_d`, `Linkedin`, `Facebook`,
   `Twitter` — come `nchar(x) > 0`.
3. **parsa cinque colonne data** e costruisce `FiscalDate` da `FiscalPeriod`
   (`"TTM 4Q2024"` → quarto trimestre → mese 12 → `2024-12-30`).
4. **conta gli affiliati** da `CompanyAffiliateRelation`: quanti in totale,
   quanti genitori, sorelle, controllate.
5. **costruisce lo scheletro del panel**: per ogni azienda una riga per ogni
   anno da `YearFounded` fino a `MaxYear`, dove `MaxYear` è l'anno più recente
   fra sei date disponibili. È qui che il dataset passa da una riga per azienda
   a una riga per azienda-anno.
6. **aggancia le variabili che variano nel tempo** allo scheletro, con cinque
   join su `(azienda, anno)`: stato di finanziamento, stato di business, stato
   di proprietà, ultima valutazione nota, e i sei dati di bilancio.
7. **filtra `YearFounded > 1999`**, che porta le aziende a 116.920.

**Produce tre file.** `db_master_1_v1.parquet` è la sezione trasversale,
`db_master_2_skeleton.parquet` lo scheletro del panel, e `db1.parquet` contiene
solo `CompanyID` e `YearFounded` **non filtrati**: serve perché alla riga 448
l'R aggancia `YearFounded` alla tabella del team prendendolo dalla versione non
filtrata, e usare quella filtrata farebbe sparire le persone delle aziende più
vecchie.


### Trappole

**Il momento della sostituzione NA.** `nchar(x) > 0` gira *dopo* la
sostituzione, quindi un URL mancante non dà `False` ma **NA**. Nel riferimento
`Website_d` vale infatti solo `True` oppure vuoto, mai `False`. Tradurlo come
«se manca allora False» sembra innocuo e cambia 9.664 righe.

**`seq()` in R conta all'indietro.** Se `MaxYear < YearFounded` — succede quando
una data è sporca — `seq(2010, 2008)` produce `2010, 2009, 2008`, cioè `Delta`
**negativi**. È il bug **B6**: 245 righe su 106 aziende. Lo riproduciamo
generando un intervallo decrescente; il flag `fix_negative_delta` emette invece
solo l'anno di fondazione.

**`pmax(..., na.rm = TRUE)`** su sei anni: se sono tutti mancanti il risultato è
NA e l'azienda viene scartata dallo scheletro.


In [ ]:
stage1_company.run(cfg)
gc.collect()

m1 = carica("db_master_1_v1")
scheletro = carica("db_master_2_skeleton")
print(f"db_master_1: {m1.height:,} righe x {m1.width} colonne   (attese 116.920 aziende)")
print(f"aziende uniche: {m1['CompanyID'].n_unique():,}")
print(f"scheletro  : {scheletro.height:,} righe azienda-anno")
print(f"anno di fondazione: da {m1['YearFounded'].min()} a {m1['YearFounded'].max()}")


#### Ispezione: il flag di presenza non è mai False

Se qui comparisse `False`, la sostituzione NA sarebbe stata applicata nel punto
sbagliato.


In [ ]:
print(m1["Website_d"].value_counts(sort=True))
print(m1["Linkedin"].value_counts(sort=True))


#### Ispezione: i Delta negativi del bug B6

Devono essere **245 righe su 106 aziende**. Sono anni-azienda *precedenti* alla
fondazione dell'azienda, generati dall'intervallo decrescente.


In [ ]:
negativi = scheletro.filter(pl.col("Delta") < 0)
print(f"righe con Delta < 0: {negativi.height}  su {negativi['CompanyID'].n_unique()} aziende")
negativi.select("CompanyID", "YearFounded", "Year_Delta", "Delta").head(6)


#### Verifica parziale

`db_master_1` non ha ancora un checkpoint suo: le sei colonne competitor
arrivano allo stadio 3. Le altre 28 però sono già definitive, e le confrontiamo
subito: aspettare la fine per scoprire un errore nato all'inizio sarebbe uno
spreco. Le sei colonne mancanti si dichiarano **assenti attese**, così non
vengono contate come errore.


In [ ]:
from src.panel.validate import load_reference, verify  # confronto in-process, solo qui

_attese = {
    "SimilarityScoreMean", "SimilarityScoreMax", "N_Competitors",
    "Same_Country", "N_Europe", "N_Outside_Europe",
}
_rapporto = verify(
    m1,
    load_reference(cfg, "db_master_1.csv"),
    key=["CompanyID"],
    name="db_master_1 (parziale, dopo lo stadio 1)",
    expected_missing=_attese,
    rtol=cfg.rtol,
)
print(_rapporto.render())
del _rapporto, m1, scheletro, negativi
gc.collect()


---
## Stadio 2a — la tabella persona-azienda (`db3`) · CHECKPOINT A

**Corrisponde a** `1_Arrange_DB.R:240-524`. Produce 534.851 righe, una per
coppia `(azienda, persona)`, ed è il primo punto con un file di riferimento
dedicato.

1. **Deduplica** `CompanyBoardTeamRelation` per `(azienda, persona)`,
   ricomponendo le righe duplicate.
2. **Aggancia 16 attributi da `Person.csv`** (956 MB, letto proiettando solo le
   colonne che servono) e ne ricava tre indici di esperienza — posizioni,
   incarichi nei board, altri ruoli — standardizzati con `scale(log(x + 1))`.
   La standardizzazione è **globale su tutta la tabella**, non per azienda:
   confrontarla per gruppo darebbe numeri completamente diversi.
3. **Aggancia l'istruzione** da `PersonEducationRelation`: classifica il titolo
   in cinque livelli e il campo di studi in nove aree, poi aggrega per persona
   (il titolo più alto, l'anno di laurea più antico, l'elenco degli atenei).
   Le catene `case_when` dell'R sono a **corto circuito** — il primo match
   vince — quindi l'ordine dei test è vincolante.
4. **Override dal nome della persona**: se il nome contiene `Ph.D`, ` JD` o
   ` MD`, il titolo più alto viene forzato a 5, e negli ultimi due casi si
   accendono anche `Is_Law` e `Is_Med`.
5. **Aggancia `PositionLevel`** da `PersonPositionRelation` e ne ricava
   `IsFounder`.
6. **Imputa la finestra di permanenza** `StartDate`/`EndDate` in quattro
   passaggi, e da lì `DeltaStart`/`DeltaEnd`: in quali anni di vita
   dell'azienda quella persona era presente.


### Trappole

**`if_else` di dplyr restituisce NA se la condizione è NA.** Alla riga 452 la
condizione è `is.na(StartDate) | year(StartDate) < YearFounded`. Per un'azienda
senza `YearFounded` il confronto vale NA, quindi l'intera condizione vale NA, e
`if_else` **cancella una StartDate perfettamente valida**. `pl.when` invece
tratta una condizione nulla come falsa e terrebbe il valore: sono **5.357
righe**, ed erano l'unica divergenza al primo tentativo su questo checkpoint.
La primitiva `rutils.r_if_else` riproduce il comportamento R.

**`PermanenzaMedia` è un solo numero** (bug **B5**). L'R la calcola con un
`summarise` *senza* `group_by`, nonostante il commento dichiari «per ciascuna
CompanyID»: è la permanenza media su tutto il dataset, usata per imputare
l'`EndDate` di chiunque. Misurato: con la media per azienda il `DeltaEnd` medio
passa da 11,57 a 12,04 anni e le righe con dati di team crescono dello 0,84%.

**`Is_Out` resta NA per le aziende non fallite** (bug **B10**). Dopo il left
join, la condizione `Is_Out == FALSE` vale NA e neutralizza una delle
imputazioni di `EndDate`. Un catch-all successivo recupera i casi, quindi il
bug si auto-sana, ma va riprodotto o i valori intermedi divergono.

**`Is_Other` cerca un valore che non esiste** (bug **B2**): confronta `Field`
con `"Other/Unknown"`, mentre `Field` produce `"Other"` e mai
`"Other/Unknown"`. La colonna è quindi falsa su tutte le righe valorizzate.

**`paste` con NA produce la stringa `"NA"`.** `IsFounder` nasce da
`str_detect(paste(FullTitle, PositionLevel, sep = "; "), "Found")`: siccome
`paste` stringifica i mancanti, la stringa non è mai NA e `IsFounder` non è mai
nullo — è sempre `True` o `False`.

**La deduplica ha due anomalie** (bug **B9**): i duplicati si riselezionano
filtrando per `PersonID` invece che per la coppia, e `coalesce(first(x),
last(x))` non vede un valore presente solo in una riga intermedia.


In [ ]:
stage2_team.run_db3(cfg)
gc.collect()

db3 = carica("db3", ["CompanyID", "PersonID", "IsFounder", "Is_Out", "Is_Other",
                     "Highest_Degree", "WorkExperienceIndex", "DeltaStart", "DeltaEnd"])
print(f"db3: {db3.height:,} righe   (attese 534.851)")
print(f"coppie (azienda, persona) uniche: {db3.select('CompanyID', 'PersonID').n_unique():,}")
print(f"founder: {db3['IsFounder'].sum():,}   IsFounder nullo: {db3['IsFounder'].null_count()}")
print(f"Is_Out nullo (bug B10): {db3['Is_Out'].null_count():,}")
print(f"Is_Other vero (bug B2): {db3['Is_Other'].sum()}")


#### Ispezione: la StartDate cancellata dall'`if_else`

Queste sono le righe in cui l'azienda non ha `YearFounded`. In R la
`StartDate` viene azzerata anche quando il dato c'era, e con essa
`DeltaStart`/`DeltaEnd`. Se qui vedessimo delle date, avremmo tradotto
l'`if_else` come un `when/otherwise`.


In [ ]:
_senza_anno = carica("db3", ["CompanyID", "PersonID", "StartDate", "EndDate",
                             "YearFounded", "IsFounder"]).filter(
    pl.col("YearFounded").is_null()
)
print(f"righe di aziende senza YearFounded: {_senza_anno.height:,}")
print(f"di queste, con StartDate valorizzata: {_senza_anno['StartDate'].is_not_null().sum()}")
_senza_anno.head(5)


#### CHECKPOINT A

Il confronto è con `db3.csv`, allineato per `(CompanyID, PersonID)`. Vanno
guardate tre cose nel rapporto: le righe attese contro quelle ottenute, le
chiavi presenti da un solo lato, e il numero di colonne divergenti. Se una
colonna divergesse, il rapporto la nomina e stampa le prime dieci righe
incriminate con la loro chiave, così il caso singolo si rifà a mano.


In [ ]:
del db3, _senza_anno
gc.collect()
verifica("A")


---
## Stadio 2b — le colonne di team, anno per anno

**Corrisponde a** `1_Arrange_DB.R:527-664`.

`db3` dice per quali anni ciascuna persona era in azienda. Questo stadio
espande quell'informazione a una riga per `(azienda, anno)` e aggrega 23
colonne di team: quante persone, quota di donne, quali aree di studio sono
rappresentate, il titolo più alto e quello medio, gli indici di esperienza,
quanti founder.

Poi fa un **full join** con lo scheletro dello stadio 1. È un full join e non
un left join perché il panel del team può contenere anni-azienda che lo
scheletro non ha: il risultato sono le **1.001.625 righe** che i checkpoint C e
D si aspettano.


### Trappole

**Il filtro di fondazione cambia soglia** (bug **B1**, il più grave del
registro). Lo stadio 1 filtra `YearFounded > 1999`; questo stadio filtra
`YearFounded > 2000`, e così fa lo stadio 4 sui deal. Risultato: **l'intera
coorte di aziende fondate nel 2000 arriva nel panel senza nessun dato di
team**, e siccome `preprocessing.py` a valle scarta le righe con
`Total_People` nullo, quella coorte **spa­risce silenziosamente** dal dataset
finale. Il flag è `fix_founding_year_threshold`.

**Le righe fantasma non esistono, e non per fortuna.** L'aggregazione conta le
persone con `.N`, che conterebbe anche la riga vuota prodotta da un join senza
corrispondenze, dando `Total_People = 1` con tutti i campi della persona nulli.
Non succede perché `YearFounded` arriva dal lato persona: per un anno-azienda
che non ha trovato nessuno vale NA, e il filtro `> 2000` elimina la riga prima
che l'aggregazione la veda. È strutturale, non un caso.

**`paste(unique(Institute), collapse = "; ")` include i NA come testo** (bug
**B7**): nel riferimento 390.544 righe hanno un `Institute` che comincia con
`"NA; "`. È cosmetico — non altera il flag «ateneo fra i primi 50», che cerca
nomi di università — ma va riprodotto, e crea un problema di confronto:
l'export del riferimento ha trasformato in null le celle che valevano *solo*
`"NA"`, quindi su quelle il confronto non può distinguere. Il motore di
verifica lo dichiara invece di nasconderlo.

**`max()` su un gruppo tutto-NA dà `-Inf` in R, `mean()` dà `NaN`.** Il ciclo
di sostituzione della riga 650 include anche `"Inf"` e `"-Inf"`, e li converte
in NA veri: è per questo che quel ciclo gira *dopo* l'aggregazione e non prima.


In [ ]:
stage2_team.run_panel(cfg)
gc.collect()

panel = carica("db_master_2_team", ["CompanyID", "YearFounded", "Year_Delta", "Delta",
                                    "Total_People", "Total_Founders", "Percent_Females"])
print(f"panel del team: {panel.height:,} righe   (attese 1.001.625)")
print(f"righe con dati di team: {panel['Total_People'].is_not_null().sum():,}")


#### Ispezione: la coorte 2000 svuotata dal bug B1

Tutte le righe delle aziende fondate nel 2000 devono avere `Total_People`
nullo. Le coorti adiacenti no.


In [ ]:
for anno in (1999, 2000, 2001, 2002):
    _c = panel.filter(pl.col("YearFounded") == anno)
    if _c.height == 0:
        print(f"coorte {anno}: assente dal panel")
        continue
    _quota = 100 * _c["Total_People"].is_not_null().sum() / _c.height
    print(f"coorte {anno}: {_c.height:>7,} righe, con dati di team {_quota:5.1f}%")


#### Ispezione: nessuna riga fantasma

Una riga con `Total_People = 1` e tutti gli altri campi di team nulli
significherebbe che il join vuoto è stato contato come una persona.


In [ ]:
_fantasma = carica(
    "db_master_2_team", ["Total_People", "Percent_Females", "Total_Founders"]
).filter(
    (pl.col("Total_People") == 1)
    & pl.col("Percent_Females").is_null()
    & pl.col("Total_Founders").is_null()
)
print(f"righe fantasma: {_fantasma.height}   (deve essere 0)")


#### Dove vengono verificate queste colonne

Le 23 colonne di team sono **definitive**: nessuno stadio successivo le
riscrive. Non le confrontiamo qui perché il confronto richiede di leggere
`db_master_2.csv` per intero come testo, qualche gigabyte, proprio nel punto in
cui l'espansione del team ha già occupato il picco di memoria della pipeline.

Sono verificate al **checkpoint C**, che confronta tutte le 107 colonne di
`db_master_2` in una volta, in un sottoprocesso. Se una colonna di team fosse
sbagliata, è lì che si vedrebbe, col nome della colonna e le prime dieci righe
divergenti.


In [ ]:
del panel, _fantasma
gc.collect()


---
## Stadio 3a — le colonne competitor · CHECKPOINT B

**Corrisponde a** `1_Arrange_DB.R:681-708`.

`CompanySimilarRelation.csv` (836 MB, 1.340.950 righe) elenca per ogni azienda
le aziende simili, con un punteggio di similarità e un flag che dice se sono
considerate concorrenti. Questo stadio lo aggrega in sei colonne per azienda e
completa `db_master_1`.

La relazione è **orientata**: «l'azienda A dichiara B come simile». La
direzione inversa non viene aggiunta, né qui né allo stadio 7.


### Trappole

**Tre aggregati sono chiamati senza `na.rm`.** `mean(SimilarityScore)`,
`max(SimilarityScore)` e `sum(IsCompetitor == "Yes")` diventano NA se *un solo*
valore manca. In polars gli aggregati saltano i null per default, quindi
darebbero un numero dove l'R dà NA. Su questi dati non scatta mai, ma il
comportamento è riprodotto comunque.

**`Same_Country` usa `any()` senza `na.rm`** (bug **B4**): restituisce NA
quando nessun confronto è vero *e* almeno uno è mancante, invece di `False`.
Sono **1.809 aziende, l'1,55% di quelle con un aggregato competitor** — poche,
ma `Same_Country` è una feature dei modelli.

**`N_Europe` e `N_Outside_Europe` non sono simmetriche** (bug **B8**):
`N_Europe` somma su *tutte* le righe, `N_Outside_Europe` solo su quelle con
similarità sopra 90. Non sono quindi due facce dello stesso conteggio.

**Il continente ha bisogno di tre stati, non due.** L'R usa `countrycode()`,
che restituisce NA per un nome che non sa collocare, e NA non è `False`:
`N_Europe` non conta né l'uno né l'altro, ma `N_Outside_Europe` conta solo
`False`. Quattro nomi cadono in quel caso — **Kosovo, Polynesia, Micronesia e
British Indian Ocean Territory** — e trattarli come «non europei» sbagliava
`N_Outside_Europe` su 44 aziende.

Per non dipendere da una libreria che può cambiare classificazione fra due
release, la mappa paese → continente è stata **ricavata dai dati** e congelata
in `src/panel/data/europe.csv`: `scripts/derive_europe_mapping.py` la deduce
per propagazione di vincoli da `N_Europe` e `N_Outside_Europe` del riferimento.
Due nomi restano indeterminati perché ogni riga che li menziona appartiene a
un'azienda fuori da `db_master_1`; è verificato che assegnarli in un modo o
nell'altro non cambia nessun output.


In [ ]:
stage3_relations.run_competitors(cfg)
gc.collect()

from src.panel.io import load_europe

_eu = load_europe()
print(f"paesi nella mappa: {len(_eu)}")
print(f"  in Europa                : {sum(1 for v in _eu.values() if v is True)}")
print(f"  fuori Europa             : {sum(1 for v in _eu.values() if v is False)}")
print(f"  continente non attribuito: {[k for k, v in _eu.items() if v is None]}")

comp = carica("db_master_1", ["CompanyID", "N_Competitors", "Same_Country",
                              "SimilarityScoreMean", "N_Europe", "N_Outside_Europe"])
print(f"\naziende: {comp.height:,}")
print(f"con aggregato competitor: {comp['N_Competitors'].is_not_null().sum():,}")
print(f"Same_Country nullo (bug B4): {comp['Same_Country'].null_count():,}")


#### CHECKPOINT B

Se questo passa, l'estrazione grezza di `CompanySimilarRelation.csv` che stiamo
usando è **la stessa su cui girava l'R**. Servirà ricordarlo allo stadio 7.


In [ ]:
verifica("B")


---
## Stadio 3b — dipendenti, financials, news

**Corrisponde a** `1_Arrange_DB.R:710-813`. Tre tabelle, tre logiche diverse:

- **dipendenti**: `CompanyEmployeeHistoryRelation` ha più rilevazioni per anno.
  L'R ordina per `(azienda, anno, data decrescente)` e tiene la prima riga di
  ogni anno, cioè **l'ultima rilevazione dell'anno**.
- **financials**: stessa deduplica su `PeriodEndDate`, ma i valori **riempiono
  solo le celle già vuote** del panel (un `coalesce`, non una sovrascrittura):
  i bilanci che lo stadio 1 aveva già agganciato da `Company.csv` vincono.
- **news**: conteggio per `(azienda, anno)`, con **0** dove il join non trova
  niente — non NA.

Nota sui dati, non sul codice: questa estrazione contiene **1.858 news su 371
aziende**, e il riferimento ne conta 1.137 su 248 righe di panel. `N_News` è
quindi zero sul 99,98% delle righe. È coerente col riferimento, ma è una
colonna praticamente vuota e va saputo prima di interpretarla come feature.


In [ ]:
stage3_relations.run_panel(cfg)
gc.collect()

rel = carica("db_master_2_relations", ["CompanyID", "Year_Delta", "EmployeeCount",
                                       "N_News", "Revenue"])
print(f"panel: {rel.height:,} righe")
print(f"N_News nullo: {rel['N_News'].null_count()}   (deve essere 0)")
print(f"N_News > 0  : {(rel['N_News'] > 0).sum():,} righe")
print(f"EmployeeCount valorizzato: {rel['EmployeeCount'].is_not_null().sum():,} righe")
print(f"Revenue valorizzato      : {rel['Revenue'].is_not_null().sum():,} righe")


#### Ispezione: i financials non sovrascrivono

Confrontiamo `Revenue` prima e dopo lo stadio: nessuna cella già valorizzata
deve essere cambiata.


In [ ]:
_prima = carica("db_master_2_team", ["CompanyID", "Year_Delta", "Revenue"])
_dopo = carica("db_master_2_relations", ["CompanyID", "Year_Delta", "Revenue"])
_j = _prima.join(_dopo, on=["CompanyID", "Year_Delta"], suffix="_dopo")
_sovrascritte = _j.filter(
    pl.col("Revenue").is_not_null() & (pl.col("Revenue") != pl.col("Revenue_dopo"))
)
_riempite = _j.filter(pl.col("Revenue").is_null() & pl.col("Revenue_dopo").is_not_null())
print(f"celle sovrascritte: {_sovrascritte.height}   (deve essere 0)")
print(f"celle riempite    : {_riempite.height:,}")

del rel, _prima, _dopo, _j, _sovrascritte, _riempite
gc.collect()


---
## Stadio 4 — deal e investitori

**Corrisponde a** `1_Arrange_DB.R:826-1251`. È lo stadio più intricato.

1. **Investitori per deal.** `DealInvestorRelation` unito a `Investor`
   classifica ogni investitore in sette categorie (venture capital, angel,
   acceleratore, corporate, private equity, investitore pubblico, altro) e
   aggrega per `DealID`: quanti investitori nuovi, le loro dimensioni medie,
   quali categorie sono presenti, quante e quali fra i lead investor.
2. **Riparazione delle date dei deal**, in quattro passaggi successivi: dallo
   stato di proprietà per fallimenti e acquisizioni, dall'anno di fondazione
   per il primo round, e infine riempiendo i buchi rimasti con la **media
   arrotondata per eccesso fra l'anno del deal precedente e quello del
   successivo**.
3. **Regola `Zero_Invested`**: i tipi di deal in cui l'importo manca in oltre
   il 90% dei casi ricevono 0 invece di NA; quelli con meno di 200 occorrenze
   vengono accorpati in `"Other"`.
4. **Quindici flag** sul tipo di deal (preseed, seed, early VC, later VC,
   M&A, uscita pubblica, fallimento, debito, private equity, grant, spin-off,
   crowdfunding, acceleratore, angel, altro).
5. **Aggregazione per `(azienda, anno)`** e innesto nel panel, più `TR_D`.


### L'imputazione RandomForest non è stata portata

Alle righe 1037-1119 l'R addestra un `randomForest(ntree = 50)` **senza
`set.seed`** e lo usa per riempire gli importi mancanti dei deal. Non è
riprodotta. Le sette colonne che creava non vengono prodotte, e gli importi
mancanti restano mancanti per l'imputazione che già gira prima del training.

Il commento in cima a `src/panel/stage4_deals.py` (`RF_SUSPENDED`) elenca le
sette colonne una per una, con la riga R in cui nascevano.

Le ragioni, oltre al seed assente che rende il risultato irriproducibile anche
rieseguendo l'R:

- **temporale**: il modello è addestrato su deal di tutti gli anni e imputa un
  deal del 2013 usando pattern del 2020, con `Age` fra i predittori;
- **train/test**: è fittato sull'intero dataset prima di qualsiasi split, e con
  lui la winsorizzazione al 95° percentile e il vincolo al terzo quartile;
- **prossimità al target**: `DealTypeGrouped` è un predittore, ma i tipi di
  deal sono ciò che determina `GrowthStage`, cioè il target.

Quanto pesava, misurato sui dataset pubblicati e non sul panel: nel dataset con
finestra temporale **5.914 righe su 30.300 (19,5%)** avevano un valore
imputato, pari al **29,7% della massa della feature**; senza finestra circa
9.739 aziende su 30.300 (32,1%). Non è un dettaglio: i risultati vanno rifatti.

La regola `Zero_Invested` appartiene alla stessa famiglia di problemi — è
derivata da statistiche globali su tutto il dataset — ma è **mantenuta**,
perché alimenta `TotalRaised`, e dire «questo tipo di deal non dichiara mai
l'importo, quindi è zero» non è un'imputazione modellistica.


### Trappole

**`pmax(year(DealDate), YearFounded)` non ha `na.rm`.** Un deal che non ha mai
ottenuto una data mantiene quindi `Year_Delta = NA`, finisce in un gruppo nullo
e **esce dal panel** al join. `pl.max_horizontal` ignora i null e restituirebbe
l'anno di fondazione, parcheggiando quei deal sull'anno zero dell'azienda e
**inventando deal in 7.335 anni-azienda**. Era la sola divergenza di questo
stadio al primo tentativo.

**Le sei varianti di `TotalRaised` hanno tre semantiche diverse** dello stesso
numero, e la differenza è il punto: `TotalRaised` è `sum(na.rm = TRUE)` (un
importo ignoto vale zero), `TotalRaised_NA` è NA se **tutti** gli importi
mancano, `TotalRaised_any` è NA se **almeno uno** manca. Le tre `_Est`
corrispondenti erano le versioni con l'imputazione.

**Gli aggregati per investitore sono condizionati a `any(InvestorStatus == "New
Investor")` senza `na.rm`**, quindi diventano NA quando nessuno corrisponde e
qualcosa manca. Il blocco dei lead investor filtra su `IsLeadInvestor` ma
conserva la stessa condizione sui nuovi investitori: l'asimmetria è nell'R e va
riprodotta.

**Il filtro `YearFounded > 2000`** torna qui: è sempre il bug B1.


In [ ]:
stage4_deals.run(cfg)
gc.collect()

deals_panel = carica("deals_panel", ["CompanyID", "Year_Delta", "N_Deal", "TotalRaised"])
print(f"deals_panel: {deals_panel.height:,} righe azienda-anno con almeno un deal")
_senza_anno = deals_panel.filter(pl.col("Year_Delta").is_null())
print(f"  di cui nel gruppo ad anno nullo (deal senza data): {_senza_anno.height:,}")
print("  quelle righe non si agganciano al panel, ed è il comportamento giusto")

from src.panel.stage4_deals import RF_SUSPENDED

p4 = carica("db_master_2_deals", ["CompanyID", "Year_Delta", "N_Deal", "TR_D",
                                  "TotalRaised", "TotalRaised_NA", "TotalRaised_any"])
print(f"\npanel: {p4.height:,} righe")
print(f"colonne della RandomForest presenti: {sorted(set(RF_SUSPENDED) & set(p4.columns))}")
print(f"TR_D = 1 (anno-azienda senza nessun deal): {(p4['TR_D'] == 1).sum():,}")
print(f"TotalRaised nullo    : {p4['TotalRaised'].null_count():,}")
print(f"TotalRaised_NA nullo : {p4['TotalRaised_NA'].null_count():,}")
print(f"TotalRaised_any nullo: {p4['TotalRaised_any'].null_count():,}")


#### Dove vengono verificate queste colonne

Cinque colonne non vengono più toccate dopo questo stadio — `DealType`,
`InvestorOwnership`, `PremoneyValuation`, `PreferredVerticals` e `TR_D` — e
sono verificate al **checkpoint C** insieme a tutte le altre. Le rimanenti
passano ancora dallo stadio 5, che le rende cumulative, quindi un confronto
fatto qui divergerebbe a ragione e non direbbe niente.

Il test `tests/panel/test_stage4.py::test_columns_final_at_stage_4_match_the_reference`
fa esattamente questo confronto in isolamento, se serve eseguirlo a parte.


In [ ]:
del deals_panel, p4, _senza_anno
gc.collect()


---
## Stadio 5 — finalizzazione · CHECKPOINT C e D

**Corrisponde a tutto** `2_Arrange_Final.R`.

1. **Ventiquattro flag diventano cumulativi.** Prima NA → `False`, poi
   `cumany` per azienda ordinata per anno: una volta che un'azienda ha fatto un
   round seed, il flag resta acceso per sempre. È quello che trasforma «in
   questo anno è successo X» in «a questo anno X era già successo».
2. **`GrowthStage`**, la variabile da cui deriva il target: una cascata di
   sette condizioni a corto circuito, quindi **l'ordine è vincolante**. Fuori
   mercato, uscita pubblica, uscita per M&A, later VC o private equity, early
   VC, seed, preseed; altrimenti NA.
3. Dove `TR_D == 1` le varianti di `TotalRaised` vanno a **0**: se l'anno non
   ha nessun deal, non ha raccolto niente.
4. **Blocco delle cumulate**: numero di deal, di round VC, capitale raccolto,
   investitori, lead investor.
5. **Medie ponderate cumulate** di quattro grandezze degli investitori, pesate
   sul numero di nuovi investitori.
6. **Attributi del CEO**: `CEO_ID` riempito in avanti, poi 18 attributi
   agganciati da `db3`. Sono invarianti nel tempo e cambiano solo quando cambia
   il CEO.
7. **Selezione delle colonne** (`db_selected`), più `StageBlock`,
   `YearsInStage`, `GrowthNextStage` e `TimeNextStage`; infine `db_final`,
   che è `db_selected` più 22 colonne di azienda.


### Trappole

**`cumsum` di R propaga NA fino in fondo al gruppo.** `cumsum(c(1, NA, 3))` è
`c(1, NA, NA)`. Quello di polars lascia un null al suo posto e **continua ad
accumulare**, dando `c(1, null, 4)`. Riguarda `TotalRaised_NA_cum` e
`TotalRaised_any_cum`, cioè proprio le due varianti che diventano NA quando un
importo non è dichiarato: la primitiva `rutils.r_cum_sum` riproduce R.

**`NewInvestors` è calcolato prima che `TotalInvestors` sia sovrascritto** dalla
propria cumulata, dentro lo stesso `mutate`. dplyr valuta in sequenza, e
invertire i due passaggi cambia silenziosamente il peso di tutte le medie
ponderate.

**`rle` tratta ogni NA come una sequenza a sé.** `YearsInStage` nasce da
`sequence(rle(GrowthStage)$lengths)`: due anni consecutivi con stadio mancante
non formano una sequenza di lunghezza due, ma due sequenze di lunghezza uno.

**`StageBlock` diventa NA per l'intera azienda** dal primo `GrowthStage`
mancante in poi (bug **B3**), perché `cumsum` su un confronto che vale NA
propaga. La colonna non è usata a valle.

**La media ponderata cumulata dell'R è O(n²)**; l'equivalente algebrico
`cumsum(x·w) / cumsum(w)` sulle sole righe valide è O(n) ed è **esatto**, non
un'approssimazione.


In [ ]:
stage5_final.run(cfg)
gc.collect()

sel = carica("db_selected", ["CompanyID", "Year_Delta", "Age", "GrowthStage",
                             "GrowthNextStage", "TimeNextStage", "YearsInStage", "N_Deal"])
print(f"db_selected: {sel.height:,} righe   (attese 1.001.625)")
print(f"db_final   : {carica('db_final', ['CompanyID']).height:,} righe (il join non deve moltiplicare)")
print("\ndistribuzione di GrowthStage:")
print(sel["GrowthStage"].value_counts(sort=True))


#### Ispezione: i flag cumulativi non si spengono

Se un'azienda avesse un flag acceso in un anno e spento in quello successivo,
la `cumany` sarebbe stata applicata male (per esempio senza ordinare, o senza
raggruppare per azienda).


In [ ]:
_flag = carica("db_master_2", ["CompanyID", "Year_Delta", "Is_Seed", "Is_EarlyVC", "Is_Out"]).sort(
    ["CompanyID", "Year_Delta"]
)
for _f in ("Is_Seed", "Is_EarlyVC", "Is_Out"):
    _spenti = _flag.filter(pl.col(_f).shift(1).over("CompanyID") & ~pl.col(_f)).height
    print(f"{_f}: righe che si spengono dopo essere state accese = {_spenti}   (deve essere 0)")


#### Ispezione: la cumulata si interrompe al primo importo ignoto

`TotalRaised_any_cum` non deve mai tornare valorizzata dopo essere diventata
nulla, all'interno della stessa azienda.


In [ ]:
_cum = carica("db_master_2", ["CompanyID", "Year_Delta", "TotalRaised_any_cum"]).sort(
    ["CompanyID", "Year_Delta"]
)
_risorte = _cum.filter(
    pl.col("TotalRaised_any_cum").is_null().shift(1).over("CompanyID")
    & pl.col("TotalRaised_any_cum").is_not_null()
).height
print(f"cumulate 'risorte' dopo un nullo: {_risorte}   (deve essere 0)")

del _cum, _flag, sel
gc.collect()


#### CHECKPOINT C e D

`db_master_2` è il panel completo, `db_selected` la sua selezione di colonne.
In entrambi i rapporti le sei colonne `_Est` compaiono come **assenti attese**
(non le produciamo) e in `db_selected` `TR_D` come **in più attesa** (l'R la
calcola e poi la butta, noi la teniamo). Tutto il resto deve coincidere
esattamente.


In [ ]:
verifica("C")


In [ ]:
verifica("D")


---
## Stadio 6 — raggruppamento degli stadi e troncamento · CHECKPOINT E

**Non corrisponde a nessuno script.** Il codice che produceva questo passaggio
è andato perduto: esisteva solo il suo output, `db_master_panel.csv.gz`. Le
quattro regole qui sotto sono state **ricostruite da quel file** e ognuna
verificata contro di esso a divergenza zero su tutte le 882.324 righe. Sono
quindi la specifica, non una congettura.

**R1 — raggruppamento degli stadi.** `GrowthStage` collassa in quattro gruppi,
preservando i nulli: `Preseed`, `Seed` e `EarlyVC` → **Early**;
`LaterVC_or_Other` → **Later**; `Out` → **Out**; `Exit_M&A` e `Exit_Public` →
**Exit**.

**R2 — stadio futuro e distanza, calcolati sulla sequenza NON troncata.** Per
ogni riga si cerca in avanti il primo gruppo diverso e non nullo, e si registra
la distanza in righe. Questo va fatto **prima** del troncamento della regola
R3: dopo, `Out` ed `Exit` non sarebbero più raggiungibili come stadio futuro, e
il senso della colonna è esattamente quello. Se non esiste un gruppo futuro
diverso, il valore è la stringa letterale `"Stay"` e la distanza è quella
dall'**ultima riga non troncata** dell'azienda. Se il gruppo corrente è nullo
non c'è mai corrispondenza, il che riproduce la semantica `NA != x` dell'R.

**R3 — troncamento.** Si eliminano tutte le righe dalla prima riga terminale
(`Out` o `Exit`) in avanti, quella riga compresa. Questo elimina anche le
5.365 righe non terminali che seguono una terminale: non è un effetto
collaterale da aggirare, è quello che deve succedere.

**R4 — `YearsInStage` ricalcolata sul gruppo**, sulla tabella troncata, con la
semantica `rle` di R. Sovrascrive il valore che lo stadio 5 aveva calcolato
sullo stadio non raggruppato.

**`StageBlock` non è riproducibile, ed è dichiarato.** Il valore nel
riferimento non corrisponde né a un ricalcolo su `GrowthStage`, né a uno sul
gruppo, né al valore di `db_selected`: sono 81.954 righe. La colonna è morta —
non la legge né `preprocessing.py` né il resto — quindi la portiamo avanti
invariata e la dichiariamo come divergenza attesa, invece di inseguirla.


In [ ]:
stage6_panel.run(cfg)
gc.collect()

mp = carica("db_master_panel", ["CompanyID", "Year_Delta", "Age", "GrowthStage",
                                "GrowthStageGroup", "GrowthNextStageGroup",
                                "TimeNextStageGroup", "YearsInStage"])
print(f"panel troncato: {mp.height:,} righe   (attese 882.324)")
print(f"aziende: {mp['CompanyID'].n_unique():,}   (attese 116.327)")
print(f"righe eliminate dal troncamento: {1_001_625 - mp.height:,}")
print("\nR1, dallo stadio al gruppo:")
print(mp.group_by("GrowthStage", "GrowthStageGroup").len().sort("len", descending=True))


#### Ispezione: nessuno stadio terminale sopravvive, ma resta raggiungibile

`GrowthStageGroup` non deve mai valere `Out` o `Exit` — quelle righe sono
troncate. `GrowthNextStageGroup` invece sì, e spesso: è la prova che R2 è stata
calcolata prima di R3.


In [ ]:
print("GrowthStageGroup (stadio corrente):")
print(mp["GrowthStageGroup"].value_counts(sort=True))
print("\nGrowthNextStageGroup (stadio futuro):")
print(mp["GrowthNextStageGroup"].value_counts(sort=True))


#### Ispezione: un'azienda passo per passo

Utile per leggere le quattro regole su un caso concreto. `100026-46` entra in
seed, passa a early VC, poi a later VC, e infine esce: la riga dell'uscita e
tutto ciò che segue non ci sono più, ma `GrowthNextStageGroup` la vede.


In [ ]:
mp.filter(pl.col("CompanyID") == "100026-46").sort("Year_Delta").drop("CompanyID")


#### CHECKPOINT E

Il riferimento di questo stadio è l'unico scritto con `write.csv` di R, dove un
valore mancante e la stringa letterale `"NA"` sono gli stessi sei byte e non si
possono distinguere. Il motore di verifica lo sa, lo dichiara, e conta a parte
le celle su cui il confronto è ambiguo.


In [ ]:
del mp
gc.collect()
verifica("E")


---
## Stadio 7 — competitor temporizzati · CHECKPOINT F

**Corrisponde a** `notebook_temporizzazione_competitors.ipynb`, celle 4 e 6.

L'R conta i competitor **una volta sola**, su tutta la lista di aziende simili,
e attacca lo stesso numero a ogni anno del panel. Questo stadio lo sostituisce
con un conteggio dei competitor **effettivamente attivi in ciascun anno**: un
concorrente fondato nel 2015 non può competere nel 2010.

Per ogni azienda si ricava una finestra di vita `[YearFounded, MaxYear]` e un
concorrente conta in un anno solo se quell'anno cade dentro la sua finestra.
Le stesse aggregazioni vengono calcolate anche **senza** il filtro temporale
(le colonne `_All`), per l'esperimento che non usa la finestra.

Infine si sostituisce `"Stay"` con il gruppo corrente, si eliminano `N_Europe`
e `N_Outside_Europe`, e si rimappano i `CompanyID` a interi consecutivi.
**Quest'ultima operazione è l'ultima della pipeline** perché distrugge ogni
possibilità di join con i file di riferimento.


### Tre colonne cambiano significato mantenendo il nome

Va saputo prima di leggerle come se fossero la stessa variabile:

- **`Same_Country`** era un booleano — «esiste un concorrente sopra 90 di
  similarità nel nostro paese» — e diventa un **conteggio** di concorrenti
  attivi nello stesso paese.
- **`SimilarityScoreMean`** viene riempita con **0** dove nessuna azienda
  simile era attiva quell'anno. Zero è il *minimo* della scala, non un valore
  neutro: un'azienda senza concorrenti vivi appare a un modello come
  un'azienda i cui concorrenti sono massimamente diversi.
- **`N_Competitors_All`** non è il `N_Competitors` dell'R: conta solo i
  concorrenti che hanno una finestra di vita utilizzabile, quindi è il minore
  dei due.

Una quarta avvertenza è metodologica e riguarda il paper, non il codice:
`MaxYear` è **l'ultimo anno con dati**, non l'anno in cui l'azienda è morta.
Un'azienda ben coperta da PitchBook risulta quindi viva più a lungo di una
coperta male, e la temporizzazione sovrappesa sistematicamente i concorrenti
grandi.

Una scelta di traduzione: le date passano da `rutils.parse_date_r`, non dal
parser del notebook originale. Il notebook leggeva `%m/%d/%Y` con fallback
`%m/%d/%y`, e chrono interpreta un anno a due cifre `25`-`69` come 2025-2069
mentre R con `cutoff_2000 = 24` lo interpreta come 1925-1969. Su questa
estrazione tutte le date hanno quattro cifre, quindi la differenza è latente e
non attiva; è stata comunque uniformata, perché è la colonna che decide se un
concorrente è vivo e due convenzioni diverse nella stessa pipeline sono un
problema che aspetta di succedere.


In [ ]:
stage7_competitors.run(cfg)
gc.collect()

finale = carica("panel", ["CompanyID", "Year_Delta", "N_Competitors", "Same_Country",
                          "SimilarityScoreMean", "N_Competitors_All",
                          "GrowthStageGroup", "GrowthNextStageGroup"])
print(f"panel finale: {finale.height:,} righe x {carica('panel', None).width} colonne")
print(f"aziende: {finale['CompanyID'].n_unique():,}   CompanyID da {finale['CompanyID'].min()} a {finale['CompanyID'].max()}")
print(f"\n'Stay' residui: {(finale['GrowthNextStageGroup'] == 'Stay').sum()}   (deve essere 0)")
print("\nconcorrenti attivi per anno-azienda:")
print(finale["N_Competitors"].describe())


#### Ispezione: temporizzato contro statico

Il conteggio temporizzato deve essere minore o uguale a quello statico: un
concorrente attivo in un dato anno è per definizione anche un concorrente.


In [ ]:
_conf = finale.select(
    (pl.col("N_Competitors") <= pl.col("N_Competitors_All")).alias("coerente")
)
print(f"righe con temporizzato <= statico: {_conf['coerente'].sum():,} su {finale.height:,}")
print(f"media temporizzata: {finale['N_Competitors'].mean():.2f}")
print(f"media statica     : {finale['N_Competitors_All'].mean():.2f}")


#### CHECKPOINT F — e il suo limite, che è nei dati e non nel codice

107 colonne su 114 riproducono esattamente il panel pubblicato. **Le sei
colonne competitor no, e la causa è accertata**: quel panel le ha calcolate da
un download di `CompanySimilarRelation.csv` diverso da quello che abbiamo.

La prova non è indiziaria:

1. il **checkpoint B** riproduce esattamente tutti gli aggregati competitor di
   `db_master_1.csv` dalla nostra estrazione, quindi la nostra estrazione è
   quella su cui girava l'R;
2. eppure **655.869 righe su 84.138 aziende** hanno lo **stesso numero di
   concorrenti** del panel di riferimento e una **media di similarità diversa**,
   in entrambe le direzioni: stesse aziende, punteggi diversi;
3. sull'azienda `100063-00` la media di riferimento 97,315 **non è la media di
   nessun sottoinsieme** dei suoi dieci punteggi nel nostro file.

I punteggi di similarità sono output di un modello che PitchBook ricalcola fra
un download e l'altro. Non è correggibile dal codice: farli combaciare
significherebbe adattare la traduzione a dati che non abbiamo. Le sei colonne
sono dichiarate in `validate.COMPETITOR_VINTAGE_COLUMNS`, con la motivazione,
e il rapporto le stampa a ogni esecuzione invece di nasconderle.


In [ ]:
verifica("F")


---
## Il registro dei difetti dell'R

Dieci difetti noti, tutti riprodotti per default. Ogni riga della tabella
corrisponde a un flag `fix_*` di `PanelConfig` che vale `False`, dove `False`
significa «comportamento R». Accenderne uno cambia il risultato e quindi rompe
i checkpoint: sono lì per la fase di correzione, non per questa.

| id | flag | cosa succede | impatto misurato |
|---|---|---|---|
| **B1** | `fix_founding_year_threshold` | il panel del team e i deal filtrano `YearFounded > 2000`, il resto `> 1999` | **ALTO.** La coorte 2000 arriva senza dati di team e a valle viene scartata: un'intera coorte di fondazione sparisce in silenzio |
| **B5** | `fix_permanenza_media_per_company` | `PermanenzaMedia` è un `summarise` senza `group_by`, quindi un unico numero globale | BASSO. `DeltaEnd` medio 11,57 contro 12,04 anni; righe con team +0,84%; `Total_Founders` +1,23% |
| **B2** | `fix_is_other_label` | `Is_Other` cerca `"Other/Unknown"`, che `Field` non produce mai | MEDIO. La colonna è falsa su tutte le righe valorizzate: inutilizzabile. Non è fra le feature dei modelli |
| **B4** | `fix_same_country_narm` | `any()` senza `na.rm` dà NA invece di `False` | BASSO come volume — 1.809 aziende, 1,55% — **ma `Same_Country` è una feature dei modelli** |
| **B3** | `fix_stageblock_na` | `cumsum` su un confronto NA propaga: `StageBlock` diventa NA per l'intera azienda | BASSO. La colonna non è usata a valle |
| **B8** | `fix_europe_asymmetry` | `N_Europe` somma su tutte le righe, `N_Outside_Europe` solo sopra 90 di similarità | BASSO. Nessuna delle due è feature dei modelli |
| **B6** | `fix_negative_delta` | se `MaxYear < YearFounded`, `seq()` conta all'indietro e genera `Delta` negativi | TRASCURABILE. 245 righe su 106 aziende |
| **B7** | `fix_institute_na_literal` | `paste(unique(Institute))` include i NA come testo `"NA"` | COSMETICO. Non altera il flag «ateneo fra i primi 50» |
| **B9** | `fix_dup_coalesce` | la deduplica filtra per `PersonID` invece che per la coppia, e `coalesce(first, last)` salta le righe intermedie | BASSO sul risultato |
| **B10** | `fix_is_out_na` | `Is_Out` resta NA per le aziende non fallite e neutralizza due imputazioni di `EndDate` | TRASCURABILE. Un catch-all successivo recupera i casi |


In [ ]:
print("stato dei flag in questa esecuzione:\n")
for _f in BUG_FLAGS:
    _v = getattr(cfg, _f)
    print(f"  {_f:<40} {'CORREZIONE ATTIVA' if _v else 'comportamento R'}")


---
## Riepilogo

| checkpoint | stadio | riferimento | righe | esito |
|---|---|---|---|---|
| A | 2a | `db3.csv` | 534.851 | 52/52 colonne identiche |
| B | 3a | `db_master_1.csv` | 116.920 | 34/34 identiche |
| C | 5 | `db_master_2.csv` | 1.001.625 | 107/107 identiche |
| D | 5 | `db_selected.csv` | 1.001.625 | 89/89 identiche |
| E | 6 | `db_master_panel.csv.gz` | 882.324 | 112/113 identiche |
| F | 7 | `data/raw/panel.csv.gz` | 882.324 | 107/114 identiche |

Tre gruppi di colonne sono **dichiarati** invece che riprodotti, e ogni
esecuzione li stampa:

1. **le sei `TotalRaised_Est*`**, che non produciamo perché l'imputazione
   RandomForest è sospesa (stadio 4);
2. **`StageBlock`** ai checkpoint E e F, non riproducibile e non letta da
   nessuno;
3. **le sei colonne competitor** al checkpoint F, calcolate da un altro
   download della tabella delle aziende simili (stadio 7).

Il panel finito è in `data/interim/panel.csv.gz`. Da lì
`scripts/build_datasets.py` costruisce i due dataset di modellazione.

### Cosa resta da decidere

`src/preprocessing.py` selezionava `TotalRaised_Est`, che non esiste più. Va
sostituita con `TotalRaised` (zero dove l'importo non è dichiarato) oppure con
`TotalRaised_NA` (nulla lì, così l'imputazione che gira prima del training vede
il buco). Entrambe vengono prodotte. In ogni caso i due dataset e tutti i run
vanno rigenerati: la feature del capitale raccolto cambia.

### Per rieseguire tutto senza notebook

```bash
uv run python scripts/check_extraction.py data/raw/pitchbook
uv run python scripts/build_panel.py --verify
```
